# Session 6 - Mini Project: Sequence Analyser

alright, final session. we're putting everything together.

we'll build a small but real sequence analyser from scratch. by the end of this you'll have a script that:
1. reads a FASTA file
2. calculates stats for each sequence (length, GC content, base composition)
3. filters sequences by criteria
4. saves results to CSV
5. makes a summary plot

this is the kind of thing you'd actually use in a real project.

## step 1 - the FASTA parser

we wrote this in session 4. let's use it again - that's why we made it a function

In [ ]:
def parse_fasta(filename):
    """read a FASTA file and return a dict of {id: sequence}"""
    sequences = {}
    current_id = None
    current_seq = []
    
    with open(filename, "r") as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if current_id:
                    sequences[current_id] = "".join(current_seq)
                current_id = line[1:].split()[0]
                current_seq = []
            elif line:
                current_seq.append(line.upper())
        if current_id:
            sequences[current_id] = "".join(current_seq)
    
    return sequences

print("parse_fasta function ready")

parse_fasta function ready


## step 2 - analysis functions

In [ ]:
def gc_content(seq):
    """calculate GC content as a percentage"""
    g = seq.count("G")
    c = seq.count("C")
    return (g + c) / len(seq) * 100

def base_composition(seq):
    """count each base"""
    return {
        "A": seq.count("A"),
        "T": seq.count("T"),
        "G": seq.count("G"),
        "C": seq.count("C"),
        "N": seq.count("N"),  # unknown bases
    }

def analyse_sequence(seq_id, sequence):
    """run all analyses on one sequence, return as dict"""
    bases = base_composition(sequence)
    return {
        "id":         seq_id,
        "length":     len(sequence),
        "gc_percent": round(gc_content(sequence), 2),
        "A":          bases["A"],
        "T":          bases["T"],
        "G":          bases["G"],
        "C":          bases["C"],
        "N":          bases["N"],
        "has_n":      bases["N"] > 0,
    }

print("analysis functions ready")

analysis functions ready


## step 3 - create a test dataset to work with

In [ ]:
# create a more realistic test FASTA
test_fasta = """>gene_001 dnaA DNA replication initiator protein
ATGAGTTTTATTATTAGGTGGCGGTACTGGGAATCGCCTTTACTATCGAAGCGATTGCCGATATTGGC
GATGAAGCGATTGAACGTATTGCGCGTATTCGCGATATCGATGCGATTGATGCGATGCGATCGATGCG
>gene_002 gyrA DNA gyrase subunit A
ATGAGCGATTTAGCGATTATCGATGGCGATATCGATGCGATTGATGCGATGCGATCGATGCGATGCGA
TTGCGATGCGATTGCGATCGATGCGATTGCGATGCGATTGCGATCGATGCGATTGCGATGCGATTGCG
>gene_003 recA recombinase DNA repair
ATGGCTATTCAGATCACTGATCTGGAGATGGGCGAGATTGATGCGATCGATGCGATCGATGCGATGCG
ATTGCGATGCGATTGCGATGCGATTGCGATGCGATGCGATTGCGATGCGATTGCGATGCGATTGCGAT
>gene_004 short_gene hypothetical
ATGCGT
>gene_005 ambiguous_gene contains Ns
ATGCGTNNNATCGATNNNGCATGC
>gene_006 lexA SOS response repressor
ATGAAACGCATTTTCCACATGAACGAAACGAATTTCGCGATTGCGATGCGATCGATGCGATTGCGATGCG
ATCGATGCGATTGCGATGCGATCGATGCGATTGCGATGCGATTGCGATGCGATCGATGCGATTGCGATG
"""

with open("data/genes.fasta", "w") as f:
    f.write(test_fasta)

print("test FASTA created: data/genes.fasta")

test FASTA created: data/genes.fasta


## step 4 - run the full analysis

In [ ]:
import pandas as pd

# read sequences
sequences = parse_fasta("data/genes.fasta")
print(f"loaded {len(sequences)} sequences")

# analyse all
results = []
for seq_id, seq in sequences.items():
    result = analyse_sequence(seq_id, seq)
    results.append(result)

# convert to dataframe
df = pd.DataFrame(results)
print(f"\nanalysis complete:")
print(df[["id", "length", "gc_percent", "has_n"]])

loaded 6 sequences

analysis complete:
          id  length  gc_percent  has_n
0   gene_001     140       51.43  False
1   gene_002     140       55.71  False
2   gene_003     140       53.57  False
3   gene_004       6       66.67  False
4   gene_005      24       50.00   True
5   gene_006     142       55.63  False


## step 5 - filter the sequences

in real work you'll want to remove sequences that are:
- too short (assembly artifacts)
- have too many N bases (poor quality)

In [ ]:
print(f"before filtering: {len(df)} sequences")

# filter: remove short sequences and sequences with N
df_filtered = df[
    (df["length"] >= 50) &   # at least 50 bp
    (df["has_n"] == False)    # no ambiguous bases
]

print(f"after filtering : {len(df_filtered)} sequences")
print(f"removed         : {len(df) - len(df_filtered)} sequences")
print()
print(df_filtered[["id", "length", "gc_percent"]])

before filtering: 6 sequences
after filtering : 4 sequences
removed         : 2 sequences

         id  length  gc_percent
0  gene_001     140       51.43
1  gene_002     140       55.71
2  gene_003     140       53.57
5  gene_006     142       55.63


## step 6 - save results to CSV

In [ ]:
df_filtered.to_csv("data/analysis_results.csv", index=False)
print("results saved to data/analysis_results.csv")

# verify it saved correctly
check = pd.read_csv("data/analysis_results.csv")
print(f"file has {len(check)} rows and {len(check.columns)} columns")

results saved to data/analysis_results.csv
file has 4 rows and 9 columns


## step 7 - summary plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# GC content bar chart
axes[0].bar(df_filtered["id"], df_filtered["gc_percent"], color="steelblue")
axes[0].set_title("GC content per gene")
axes[0].set_ylabel("GC content (%)")
axes[0].set_ylim(0, 100)
axes[0].axhline(y=50, color="red", linestyle="--", alpha=0.5, label="50%")
axes[0].tick_params(axis="x", rotation=30)
axes[0].legend()

# sequence length bar chart
axes[1].bar(df_filtered["id"], df_filtered["length"], color="coral")
axes[1].set_title("sequence length per gene")
axes[1].set_ylabel("length (bp)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("data/sequence_summary.png", dpi=150)
plt.show()
print("plot saved")

plot saved


## you just built a real bioinformatics script

let's look at what it does:

1. reads a FASTA file with any number of sequences
2. calculates length, GC%, base composition for each
3. filters out short or low-quality sequences
4. saves results to a CSV you can open in Excel
5. generates a summary figure

this is genuinely useful. you could use this on real sequencing output.

---

## challenge - extend the script

try adding one or more of these features:

1. a function to find ORFs (start with ATG, end with TAA/TAG/TGA)
2. filter by GC content range (e.g. only sequences with 45-65% GC)
3. add a column for AT content = 100 - GC content
4. sort the results by length (hint: `df.sort_values("length")`)

good luck and well done getting through the whole workshop!

In [ ]:
# space for your challenge code

